#Ingesta de Fraude: bronze

In [0]:
# CONFIGURACIÓN DE PARÁMETROS

dbutils.widgets.text("catalog_name", "banking_dev", "Catálogo")
dbutils.widgets.text("storage_account", "adlsprojectbank12", "Storage Account")
dbutils.widgets.dropdown("load_mode", "FULL", ["FULL"], "Modo de carga")

catalog_name = dbutils.widgets.get("catalog_name").strip()
storage_account = dbutils.widgets.get("storage_account").strip()
load_mode = dbutils.widgets.get("load_mode").strip().upper()

if not catalog_name or not storage_account:
    raise ValueError("catalog_name y storage_account son obligatorios.")
if load_mode != "FULL":
    raise ValueError(f"Modo de carga no soportado: {load_mode}")

print(f"Catálogo: {catalog_name}")
print(f"Storage: {storage_account}")
print(f"Modo de carga: {load_mode}")

In [0]:
# CONFIGURACIÓN DE RUTAS

source_path = (f"abfss://raw@{storage_account}.dfs.core.windows.net/fraud/creditcard.csv")
target_table = f"{catalog_name}.bronze.fraud_transactions"
target_path = (f"abfss://bronze@{storage_account}.dfs.core.windows.net/fraud_transactions")

print(f"Origen : {source_path}")
print(f"Destino: {target_table}")
print(f"Delta  : {target_path}")

In [0]:
# DEFINIR ESQUEMAS

from pyspark.sql.types import (
    StructType, StructField, DoubleType, DecimalType, IntegerType
)

feature_columns = [f"V{i}" for i in range(1, 29)]

fraud_schema = StructType(
    [StructField("Time", DoubleType(), True)]
    + [StructField(c, DoubleType(), True) for c in feature_columns]
    + [
        StructField("Amount", DecimalType(18, 2), True),
        StructField("Class", IntegerType(), True)
    ]
)

print(f"Columnas esperadas en Raw: {len(fraud_schema.fields)}")
print(fraud_schema.simpleString())

In [0]:
# LECTURA DE ARCHIVOS CSV

df_fraud = (
    spark.read.format("csv")
    .schema(fraud_schema)
    .option("header", "true")
    .option("encoding", "UTF-8")
    .option("sep", ",")
    .option("mode", "FAILFAST")
    .load(source_path)
)

source_count = df_fraud.count()
print(f"Columnas: {len(df_fraud.columns)}")
print(f"Registros: {source_count:,}")

if source_count == 0:
    raise ValueError("El archivo de fraude no contiene registros.")

display(df_fraud.limit(10))

In [0]:
# INCORPORACION DE INGESTION_DATE

from pyspark.sql import functions as F

df_fraud = df_fraud.withColumn(
    "ingestion_date",
    F.current_timestamp()
)

if "ingestion_date" not in df_fraud.columns:
    raise ValueError("No se agregó ingestion_date.")

print(f"Columnas para Bronze: {len(df_fraud.columns)}")

In [0]:
# VALIDAR DATASETS

from pyspark.sql import functions as F

# Nulos por columna
null_counts = df_fraud.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df_fraud.columns
]).first().asDict()

total_nulls = sum(null_counts.values())
print("VALORES NULOS POR COLUMNA")
for column, count in null_counts.items():
    print(f"{column:18} | {count:>10,}")

if total_nulls > 0:
    raise ValueError(
        f"Se encontraron {total_nulls:,} valores nulos. Revisar el CSV antes de cargar Bronze."
    )

# Dominio de Class
invalid_class_count = df_fraud.filter(
    F.col("Class").isNull() | (~F.col("Class").isin(0, 1))
).count()

if invalid_class_count > 0:
    raise ValueError(f"Se encontraron {invalid_class_count:,} valores inválidos en Class.")

# Importe no negativo
negative_amount_count = df_fraud.filter(F.col("Amount") < 0).count()
if negative_amount_count > 0:
    raise ValueError(f"Se encontraron {negative_amount_count:,} importes negativos.")

# Distribución de clases
class_distribution = (
    df_fraud.groupBy("Class").count().orderBy("Class")
)
print("DISTRIBUCIÓN DE CLASES")
display(class_distribution)

class_counts = {r["Class"]: r["count"] for r in class_distribution.collect()}
legitimate_count = class_counts.get(0, 0)
fraud_count = class_counts.get(1, 0)

print(f"Transacciones legítimas : {legitimate_count:,}")
print(f"Transacciones fraudulentas: {fraud_count:,}")
print(f"Total: {legitimate_count + fraud_count:,}")

if legitimate_count + fraud_count != source_count:
    raise ValueError("La distribución de Class no coincide con el total leído.")

print("Validaciones de calidad completadas.")

In [0]:
# CARGA DE DATOS FULL A TABLAS BRONZE

target_columns = spark.table(target_table).columns
df_to_load = df_fraud.select(*target_columns)
expected_count = df_to_load.count()

print(f"Cargando {target_table}...")
(
    df_to_load.write
    .mode("overwrite")
    .insertInto(target_table)
)

print(f"Registros enviados: {expected_count:,}")
print("Escritura completa")

In [0]:
# VALIDAR CARGA

df_bronze = spark.table(target_table)
actual_count = df_bronze.count()

bronze_class_counts = {
    r["Class"]: r["count"]
    for r in df_bronze.groupBy("Class").count().collect()
}
bronze_legitimate_count = bronze_class_counts.get(0, 0)
bronze_fraud_count = bronze_class_counts.get(1, 0)

count_status = "SUCCESS" if expected_count == actual_count else "FAILED"
legitimate_status = (
    "SUCCESS" if legitimate_count == bronze_legitimate_count else "FAILED"
)
fraud_status = (
    "SUCCESS" if fraud_count == bronze_fraud_count else "FAILED"
)

print(f"Registros esperados : {expected_count:,}")
print(f"Registros obtenidos : {actual_count:,}")
print(f"Validación total    : {count_status}")
print()
print(f"Legítimas esperadas : {legitimate_count:,}")
print(f"Legítimas obtenidas : {bronze_legitimate_count:,}")
print(f"Validación legítimas: {legitimate_status}")
print()
print(f"Fraudes esperados   : {fraud_count:,}")
print(f"Fraudes obtenidos   : {bronze_fraud_count:,}")
print(f"Validación fraudes  : {fraud_status}")

if "FAILED" in [count_status, legitimate_status, fraud_status]:
    raise ValueError("La validación posterior a la carga falló.")

print("Validación posterior a la carga completada.")

In [0]:
#RESUMEN

print("=" * 72)
print("RESUMEN DE INGESTA FRAUD - BRONZE")
print("=" * 72)
print(f"Catálogo                  : {catalog_name}")
print(f"Modo de carga             : {load_mode}")
print(f"Tabla procesada           : {target_table}")
print(f"Registros totales         : {actual_count:,}")
print(f"Transacciones legítimas   : {bronze_legitimate_count:,}")
print(f"Transacciones fraudulentas: {bronze_fraud_count:,}")
print(f"Validación de registros   : {count_status}")
print(f"Validación de legítimas   : {legitimate_status}")
print(f"Validación de fraude      : {fraud_status}")
print("=" * 72)

if all(s == "SUCCESS" for s in [
    count_status, legitimate_status, fraud_status
]):
    print("RESULTADO: INGESTA FRAUD COMPLETADA CORRECTAMENTE")
else:
    raise RuntimeError("La ingesta Fraud finalizó con errores.")

print("=" * 72)